# Essai 1 - Whisper-small (à la rache)

## Transcription d'un ficher audio (test):

In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 74.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.1
    Uninstalling transformers-5.20.1:
      Successfully uninstalled transformers-5.20.1


In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import Audio
import librosa
import IPython.display as ipd

# 1. Définir le chemin de ton fichier audio
audio_path = "/content/Audio/00377_clip_010.mp3"

# 2. Afficher/Écouter l'audio directement dans Google Colab
display(ipd.Audio(audio_path))

# 3. Charger le modèle et le processeur Whisper
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.config.forced_decoder_ids = None

# 4. Charger l'audio avec librosa en s'assurant d'avoir la bonne fréquence d'échantillonnage (16000 Hz requis pour Whisper)
speech_array, sampling_rate = librosa.load(audio_path, sr=16000)

# 5. Préparer les caractéristiques d'entrée (input features) pour le modèle
input_features = processor(
    speech_array,
    sampling_rate=sampling_rate,
    return_tensors="pt"
).input_features

# 6. Générer les tokens de transcription
predicted_ids = model.generate(input_features)

# 7. Décoder les tokens en texte
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)

print("Transcription :", transcription[0])

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

Transcription :  honor past heavyweight champions, warriors like Rocky Balboa. Let's hope


On a bien verifié que ce modele arrive a bien transcrire l'audio.

## Creation de l'ensamble de transcriptions des données

In [ ]:
# 1. Montage de Google Drive
from google.colab import drive
from pathlib import Path
import json
import torch
from transformers import pipeline
from tqdm.notebook import tqdm

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 2. Spécifiez le chemin vers votre dossier 'movie' sur Google Drive
# Assurez-vous d'ajuster le chemin si votre dossier est dans un sous-dossier
BASE_DIR = Path('/content/drive/MyDrive/Projet Multimedia/movie')
AUDIO_DIR = BASE_DIR / 'audio'
TRANSCRIPT_DIR = BASE_DIR / 'transcriptions'

# Vérification du GPU
device = 0 if torch.cuda.is_available() else -1
print(f"Mode d'exécution : {'GPU (' + torch.cuda.get_device_name(0) + ')' if device == 0 else 'CPU (Attention : lent)'}")

Mode d'exécution : GPU (Tesla T4)


In [ ]:
# 3. Chargement du modèle Whisper sur GPU
transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device,
)

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

... (3 outputs left out)

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

In [ ]:
# 4. Liste de tous les fichiers MP3
audio_files = sorted(list(AUDIO_DIR.glob('*/*.mp3')))
print(f"Nombre total de fichiers audio trouvés : {len(audio_files)}")

Nombre total de fichiers audio trouvés : 3088


In [ ]:
# 5. Boucle de transcription avec barre de progression
for audio_path in tqdm(audio_files, desc="Transcription en cours"):
    film_id = audio_path.parent.name  # ex: "00000"
    clip_id = audio_path.stem         # ex: "00000_clip_001"

    # Dossier cible pour le film dans transcriptions/
    target_film_dir = TRANSCRIPT_DIR / film_id
    target_film_dir.mkdir(parents=True, exist_ok=True)
    output_json_path = target_film_dir / f"{clip_id}.json"

    # Sauter si le fichier JSON existe déjà (reprise automatique)
    if output_json_path.exists():
        continue

    try:
        result = transcriber(str(audio_path))
        text = result.get("text", "").strip()
        data = {
            "video_id": clip_id,
            "transcript": text,
            "model": "whisper-small"
        }
        with open(output_json_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"\nErreur sur {clip_id}: {e}")

print("\nToutes les transcriptions ont été générées et sauvegardées sur votre Google Drive !")

Transcription en cours:   0%|          | 0/3088 [00:00<?, ?it/s]


Toutes les transcriptions ont été générées et sauvegardées sur votre Google Drive !


## Post-traitement

In [ ]:
import json
from pathlib import Path
import re
from tqdm.notebook import tqdm

# 1. Définition des chemins
BASE_DIR = Path("/content/drive/MyDrive/Projet Multimedia/movie")
TRANSCRIPT_DIR = BASE_DIR / "transcriptions"
OUTPUT_FILE = BASE_DIR / "problematic_clips2.json"

def detect_problematic_transcripts(transcript_dir: Path, output_file: Path):
    # Motif Regex pour détecter la répétition d'un mot ou groupe de 1 à 3 mots (4 fois ou plus au total)
    repetition_pattern = re.compile(
        r"\b(\w+(?:\s+\w+){0,2})\b(?:\s+\1){3,}",
        re.IGNORECASE
    )

    # Motif Regex strict pour l'anglais, l'espagnol et le français (lettres, accents, chiffres et ponctuation autorisée + $ ¡ ¿)
    non_standard_chars_pattern = re.compile(
        r"[^a-zA-ZñÑáéíóúÁÉÍÓÚüÜàâæçéèêëîïôœùûüÿÀÂÆÇÉÈÊËÎÏÔŒÙÛÜŸ0-9\s\.\,\!\?\'\-\$¡¿]",
        re.UNICODE
    )

    problematic_clips = []
    stats = {
        "total_analyzed": 0,
        "empty_or_missing": 0,
        "non_standard_chars": 0,
        "repetitions": 0,
        "total_problematic": 0,
    }

    # Parcourt tous les fichiers .json dans tous les sous-dossiers de films
    json_files = sorted(list(transcript_dir.glob("*/*.json")))
    print(f"Nombre total de fichiers JSON trouvés : {len(json_files)}")

    for json_path in tqdm(json_files, desc="Analyse des transcriptions en cours"):
        stats["total_analyzed"] += 1
        clip_id = json_path.stem      # ex: "00000_clip_001"
        film_id = json_path.parent.name # ex: "00000"

        try:
            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            transcript = data.get("transcript", "").strip()
            issues = []

            # --- Cas 1 : Pas de transcription ou texte extrêmement court / vide ---
            if not transcript or len(transcript) < 2:
                issues.append("vide_ou_incomplet")
                stats["empty_or_missing"] += 1
            else:
                # --- Cas 1.2 : Uniquement des symboles ou de la ponctuation sans mots réels ---
                if not re.search(r'[a-zA-ZÀ-ÿ0-9]', transcript):
                    issues.append("unicament_simbols_o_puntuacio")
                    stats["empty_or_missing"] += 1
                else:
                    # --- Cas 2 : Symboles en dehors de l'alphabet (ang, fr, esp), chiffres ou ponctuation standard ---
                    strange_chars = set(non_standard_chars_pattern.findall(transcript))
                    if strange_chars:
                        issues.append(f"caracteres_anormaux ({' '.join(sorted(strange_chars))})")
                        stats["non_standard_chars"] += 1

                    # --- Cas 3 : Répétition en boucle (4 fois ou plus) ---
                    if repetition_pattern.search(transcript):
                        issues.append("repetitions_en_boucle")
                        stats["repetitions"] += 1

            # Si au moins un problème est détecté, on enregistre le clip
            if issues:
                problematic_clips.append({
                    "video_id": clip_id,
                    "film_id": film_id,
                    "json_path": str(json_path),
                    "transcript_original": transcript,
                    "issues": issues,
                })

        except Exception as e:
            problematic_clips.append({
                "video_id": clip_id,
                "film_id": film_id,
                "json_path": str(json_path),
                "transcript_original": None,
                "issues": [f"erreur_lecture_json ({str(e)})"],
            })

    # Calcul des totaux uniques
    stats["total_problematic"] = len(problematic_clips)

    # Structure du rapport final
    report = {
        "statistics": stats,
        "problematic_clips": problematic_clips
    }

    # Écriture dans le fichier JSON de sortie
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    # Affichage du bilan dans la console
    print("\n" + "=" * 40)
    print(" BILAN DE L'ANALYSE")
    print("=" * 40)
    print(f"Total des clips analysés : {stats['total_analyzed']}")
    pct = (stats["total_problematic"] / stats["total_analyzed"]) * 100 if stats["total_analyzed"] > 0 else 0
    print(f"Clips problématiques détectés : {stats['total_problematic']} ({pct:.1f}%)")
    print(f" - 1. Transcriptions vides ou symboles : {stats['empty_or_missing']}")
    print(f" - 2. Caractères anormaux : {stats['non_standard_chars']}")
    print(f" - 3. Répétitions en boucle : {stats['repetitions']}")
    print(f"\nFichier récapitulatif sauvegardé dans : {output_file}")

    return report

# Lancement du diagnostic
report = detect_problematic_transcripts(TRANSCRIPT_DIR, OUTPUT_FILE)

Nombre total de fichiers JSON trouvés : 3088


Analyse des transcriptions en cours:   0%|          | 0/3088 [00:00<?, ?it/s]


 BILAN DE L'ANALYSE
Total des clips analysés : 3088
Clips problématiques détectés : 243 (7.9%)
 - 1. Transcriptions vides ou symboles : 213
 - 2. Caractères anormaux : 23
 - 3. Répétitions en boucle : 8

Fichier récapitulatif sauvegardé dans : /content/drive/MyDrive/Projet Multimedia/movie/problematic_clips2.json


In [ ]:
import re

def clean_whisper_transcript(text: str) -> str:
    if not text or not isinstance(text, str):
        return ""

    cleaned = text.strip()

    # 1. Éliminer les répétitions en boucle (ex: "Yeah Yeah Yeah Yeah" -> "Yeah")
    # Détecte un mot ou motif répété 3 fois ou plus
    cleaned = re.sub(
        r"\b(\w+(?:\s+\w+){0,2})\b(?:\s+\1){2,}",
        r"\1",
        cleaned,
        flags=re.IGNORECASE,
    )

    # 3. Supprimer les caractères aberrants / non imprimables (conserver alphabet latin, chiffres et ponctuation)
    cleaned = re.sub(r"[^\w\s\.\,\!\?\'-]", "", cleaned)

    # 4. Nettoyer les espaces multiples
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    return cleaned